# Cross-Sectional Stock Return Prediction

In [1]:
import sys
print(sys.executable)

/Users/alexisfernandez/Coding/personal/portfolio/project1/.venv/bin/python


In [2]:
import pandas as pd
import numpy as np
import yfinance as yf

In [3]:
#Chosen stocks
tickers = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "META",
    "NVDA", "JPM", "XOM", "JNJ", "PG"
]

In [4]:
#Download data
prices = yf.download(tickers, start='2015-01-01', end = '2026-01-01', auto_adjust=True)
prices.head()

[*********************100%***********************]  10 of 10 completed


Price           Close                                                       \
Ticker           AAPL     AMZN      GOOGL        JNJ        JPM       META   
Date                                                                         
2015-01-02  24.171761  15.4260  26.227924  75.737358  45.838112  77.705750   
2015-01-05  23.490801  15.1095  25.728178  75.208374  44.415066  76.457710   
2015-01-06  23.493008  14.7645  25.093220  74.838829  43.263420  75.427567   
2015-01-07  23.822432  14.9210  25.019424  76.490990  43.329449  75.427567   
2015-01-08  24.737743  15.0230  25.106594  77.092430  44.297695  77.438316   

Price                                                  ...     Volume  \
Ticker           MSFT      NVDA         PG        XOM  ...       AAPL   
Date                                                   ...              
2015-01-02  39.607174  0.481884  65.070007  56.777885  ...  212818400   
2015-01-05  39.242943  0.473745  64.760605  55.224342  ...  257142000   
2015-01-06  38.666973  0.459381  64.465645  54.930756  ...  263188400   
2015-01-07  39.158253  0.458185  64.803787  55.487366  ...  160423600   
2015-01-08  40.310215  0.475420  65.544846  56.410919  ...  237458000   

Price                                                                  \
Ticker          AMZN     GOOGL      JNJ       JPM      META      MSFT   
Date                                                                    
2015-01-02  55664000  26480000  5753600  12600000  18177500  27913900   
2015-01-05  55484000  41182000  8079300  20100600  26452200  39673900   
2015-01-06  70380000  54456000  7428000  29074100  27399300  36447900   
2015-01-07  52806000  46918000  7931700  23843200  22045300  29114100   
2015-01-08  61768000  73054000  9916000  16971100  23961000  29645200   

Price                                     
Ticker           NVDA       PG       XOM  
Date                                      
2015-01-02  113680000  7251400  10220400  
2015-01-05  197952000  8626100  18502400  
2015-01-06  197764000  7791200  16670700  
2015-01-07  321808000  5986600  13590700  
2015-01-08  283780000  6823300  15487500  

[5 rows x 50 columns]

In [5]:
close = prices["Close"] #Closing prices (daily)
prices_monthly = close.resample("ME").last() #Month's end closing prices
returns_monthly = prices_monthly.pct_change() #Calculate returns

returns_monthly.head()

Ticker,AAPL,AMZN,GOOGL,JNJ,JPM,META,MSFT,NVDA,PG,XOM
Date,,,,,,,,,,
2015-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-02-28,0.100777,0.072293,0.046656,0.030833,0.126885,0.040311,0.093120,0.153383,0.009966,0.020435
2015-03-31,-0.031372,-0.021202,-0.014095,-0.018633,-0.011423,0.041155,-0.072748,-0.051224,-0.037472,-0.039982
2015-04-30,0.005786,0.133513,-0.010690,-0.013916,0.051180,-0.041961,0.196262,0.060678,-0.021851,0.027883
2015-05-31,0.045339,0.017663,-0.006287,0.016837,0.039836,0.005332,-0.030335,0.001461,-0.014086,-0.016772


In [6]:
#target variable
future_returns = returns_monthly.shift(-1)
future_returns.head()

Ticker,AAPL,AMZN,GOOGL,JNJ,JPM,META,MSFT,NVDA,PG,XOM
Date,,,,,,,,,,
2015-01-31,0.100777,0.072293,0.046656,0.030833,0.126885,0.040311,0.093120,0.153383,0.009966,0.020435
2015-02-28,-0.031372,-0.021202,-0.014095,-0.018633,-0.011423,0.041155,-0.072748,-0.051224,-0.037472,-0.039982
2015-03-31,0.005786,0.133513,-0.010690,-0.013916,0.051180,-0.041961,0.196262,0.060678,-0.021851,0.027883
2015-04-30,0.045339,0.017663,-0.006287,0.016837,0.039836,0.005332,-0.030335,0.001461,-0.014086,-0.016772
2015-05-31,-0.037227,0.011323,-0.009682,-0.026762,0.030100,0.083091,-0.057832,-0.091279,-0.001913,-0.023474


## Feature Engineering

In [7]:
# features_df = future_returns.reset_index().melt(
#     id_vars="Date",
#     var_name="Ticker",
#     value_name="Future Returns"
# )

# features_df.head()


### Momentum

The momentum at time $t$ is given by $$ \frac{P_{t-1}}{P_{t-12}} - 1 $$

measuring how much the stock has increased or decreaed over the past year

In [8]:
momentum = prices_monthly.shift(1) / prices_monthly.shift(12) - 1

In [9]:
momentum_df = momentum.reset_index().melt(
    id_vars="Date",
    var_name="Ticker",
    value_name="Momentum"
)

features_df = momentum_df.copy()
features_df.tail()

,Date,Ticker,Momentum
1315,2025-08-31,XOM,-0.028234
1316,2025-09-30,XOM,0.010258
1317,2025-10-31,XOM,0.000400
1318,2025-11-30,XOM,-0.003652
1319,2025-12-31,XOM,0.117174


### Volatility

In [10]:
volatility = prices_monthly.rolling(12).std()

In [11]:
volatility_df = volatility.reset_index().melt(
    id_vars="Date",
    var_name="Ticker",
    value_name="Volatility"
)
features_df['Volatility'] = volatility_df['Volatility']
features_df.tail()

,Date,Ticker,Momentum,Volatility
1315,2025-08-31,XOM,-0.028234,4.983388
1316,2025-09-30,XOM,0.010258,4.959673
1317,2025-10-31,XOM,0.000400,5.077397
1318,2025-11-30,XOM,-0.003652,5.302776
1319,2025-12-31,XOM,0.117174,5.991959


### Beta

In [12]:
#Download market data
market = yf.download(
    "^GSPC",
    start="2015-01-01",
    end="2026-01-01",
    auto_adjust=True
)

market_prices = market["Close"].resample("ME").last()
market_returns = market_prices.squeeze().pct_change()

[*********************100%***********************]  1 of 1 completed


In [13]:
returns_monthly = returns_monthly.astype(float)
market_returns = market_returns.astype(float)


In [14]:
#Calculate betas
betas = {}
for ticker in tickers:
    betas[ticker] = returns_monthly[ticker].rolling(12).cov(market_returns) / market_returns.rolling(12).var()

betas

{'AAPL': Date
 2015-01-31         NaN
 2015-02-28         NaN
 2015-03-31         NaN
 2015-04-30         NaN
 2015-05-31         NaN
                 ...   
 2025-08-31    0.368902
 2025-09-30    0.493240
 2025-10-31    0.463809
 2025-11-30    0.389568
 2025-12-31    0.619201
 Freq: ME, Length: 132, dtype: float64,
 'MSFT': Date
 2015-01-31         NaN
 2015-02-28         NaN
 2015-03-31         NaN
 2015-04-30         NaN
 2015-05-31         NaN
                 ...   
 2025-08-31    1.342136
 2025-09-30    1.298322
 2025-10-31    1.202573
 2025-11-30    1.352793
 2025-12-31    1.473587
 Freq: ME, Length: 132, dtype: float64,
 'GOOGL': Date
 2015-01-31         NaN
 2015-02-28         NaN
 2015-03-31         NaN
 2015-04-30         NaN
 2015-05-31         NaN
                 ...   
 2025-08-31    0.879256
 2025-09-30    1.034016
 2025-10-31    1.122435
 2025-11-30    1.435124
 2025-12-31    1.908740
 Freq: ME, Length: 132, dtype: float64,
 'AMZN': Date
 2015-01-31         NaN
 2015-0

In [15]:
betas_predf = pd.DataFrame(betas)
betas_df = betas_predf.reset_index().melt(
    id_vars="Date",
    var_name="Ticker",
    value_name="Beta"
)

betas_df

,Date,Ticker,Beta
0,2015-01-31,AAPL,NaN
1,2015-02-28,AAPL,NaN
2,2015-03-31,AAPL,NaN
3,2015-04-30,AAPL,NaN
4,2015-05-31,AAPL,NaN
...,...,...,...
1315,2025-08-31,PG,0.552709
1316,2025-09-30,PG,0.501826
1317,2025-10-31,PG,0.454634
1318,2025-11-30,PG,0.190867


In [16]:
features_df['Beta'] = betas_df['Beta']
features_df.tail()

,Date,Ticker,Momentum,Volatility,Beta
1315,2025-08-31,XOM,-0.028234,4.983388,0.552709
1316,2025-09-30,XOM,0.010258,4.959673,0.501826
1317,2025-10-31,XOM,0.000400,5.077397,0.454634
1318,2025-11-30,XOM,-0.003652,5.302776,0.190867
1319,2025-12-31,XOM,0.117174,5.991959,0.057403


### Current Size \& Value

Note: need to be adjusted into historical data for backtesting, but temporarily uses current data

In [17]:
market_caps = {}
pe_ratios = {}

for ticker in tickers:
    info = yf.Ticker(ticker).info
    market_caps[ticker] = info.get('marketCap')
    pe_ratios[ticker] = info.get('trailingPE')

In [18]:
market_caps

{'AAPL': 5020252110848,
 'MSFT': 3957444575232,
 'GOOGL': 4094093164544,
 'AMZN': 2830004912128,
 'META': 1776936550400,
 'NVDA': 5359909470208,
 'JPM': 890130595840,
 'XOM': 732290351104,
 'JNJ': 621223608320,
 'PG': 364564086784}

In [19]:
#Use log market cap instead for feature
log_market_caps = {
    ticker : float(np.log(cap))
    for ticker, cap in market_caps.items()
    if cap is not None
}

log_market_caps

{'AAPL': 29.2445012696556,
 'MSFT': 29.006619623608437,
 'GOOGL': 29.04056635927023,
 'AMZN': 28.671299563316463,
 'META': 28.205911958442204,
 'NVDA': 29.309968200982055,
 'JPM': 27.514634025806668,
 'XOM': 27.31944292677478,
 'JNJ': 27.154956931877418,
 'PG': 26.621968194021342}

In [20]:
#P/E ratios to Earnings yield

earnings_yield = {
    ticker : 1 / pe
    for ticker, pe in pe_ratios.items()
    if pe is not None and pe > 0
}

earnings_yield

{'AAPL': 0.025349577004283315,
 'MSFT': 0.03368045873593129,
 'GOOGL': 0.059535184996657095,
 'AMZN': 0.04737584484751919,
 'META': 0.042020300763664344,
 'NVDA': 0.035635446107855594,
 'JPM': 0.07112384133039136,
 'XOM': 0.04362962531837628,
 'JNJ': 0.025564434594979084,
 'PG': 0.04220862007231772}

In [21]:
curr_df = pd.DataFrame({
    "Ticker" : list(log_market_caps.keys()),
    "Size" : log_market_caps.values(),
    "Value" : earnings_yield.values()
})

curr_df.head()

,Ticker,Size,Value
0,AAPL,29.244501,0.025350
1,MSFT,29.006620,0.033680
2,GOOGL,29.040566,0.059535
3,AMZN,28.671300,0.047376
4,META,28.205912,0.042020


In [22]:
features_df = features_df.merge(
    curr_df,
    on="Ticker",
    how="left"
)

features_df.tail()

,Date,Ticker,Momentum,Volatility,Beta,Size,Value
1315,2025-08-31,XOM,-0.028234,4.983388,0.552709,27.319443,0.04363
1316,2025-09-30,XOM,0.010258,4.959673,0.501826,27.319443,0.04363
1317,2025-10-31,XOM,0.000400,5.077397,0.454634,27.319443,0.04363
1318,2025-11-30,XOM,-0.003652,5.302776,0.190867,27.319443,0.04363
1319,2025-12-31,XOM,0.117174,5.991959,0.057403,27.319443,0.04363


In [23]:
target_df = future_returns.reset_index().melt(
    id_vars="Date",
    var_name="Ticker",
    value_name="Future Returns"
)

features_df['Future Returns'] = target_df['Future Returns']
features_df.tail()

,Date,Ticker,Momentum,Volatility,Beta,Size,Value,Future Returns
1315,2025-08-31,XOM,-0.028234,4.983388,0.552709,27.319443,0.04363,-0.013475
1316,2025-09-30,XOM,0.010258,4.959673,0.501826,27.319443,0.04363,0.014279
1317,2025-10-31,XOM,0.000400,5.077397,0.454634,27.319443,0.04363,0.022507
1318,2025-11-30,XOM,-0.003652,5.302776,0.190867,27.319443,0.04363,0.038130
1319,2025-12-31,XOM,0.117174,5.991959,0.057403,27.319443,0.04363,NaN


## Data Cleaning

remove NaN values

In [24]:
data = features_df.dropna(subset=['Momentum', 'Volatility', 'Beta', 'Size', 'Value', 'Future Returns']).copy()

print(data.shape)
print(data.head())

(1190, 8)
         Date Ticker  Momentum  Volatility      Beta       Size    Value  \
12 2016-01-31   AAPL -0.086263    2.129122  1.279009  29.244501  0.02535   
13 2016-02-29   AAPL -0.232374    2.380289  1.116992  29.244501  0.02535   
14 2016-03-31   AAPL -0.208532    2.333381  1.340058  29.244501  0.02535   
15 2016-04-30   AAPL -0.112981    2.539117  1.320437  29.244501  0.02535   
16 2016-05-31   AAPL -0.270183    2.278701  1.347972  29.244501  0.02535   

    Future Returns  
12       -0.001287  
13        0.127211  
14       -0.139921  
15        0.071773  
16       -0.042660  


split data

In [25]:
data['Date'] = pd.to_datetime(data['Date'])

train = data[data['Date'] < '2024-01-01'].copy()
test = data[data['Date'] >= '2024-01-01'].copy()

print(len(train) , len(test))

960 230


In [26]:
features = ['Momentum', 'Volatility', 'Beta', 'Size', 'Value']

X_train = train[features]
y_train = train['Future Returns']

X_test = test[features]
y_test = test['Future Returns']

## Baseline Linear Regression Model

In [27]:
from sklearn.linear_model import LinearRegression

lm = LinearRegression()
lm.fit(X_train, y_train)

preds_lm = lm.predict(X_test)
test['Predicted Returns'] = preds_lm

In [28]:
from sklearn.metrics import mean_squared_error, r2_score

rmse_lm = np.sqrt(mean_squared_error(y_test, preds_lm))
r2_lm = r2_score(y_test, preds_lm)

print('RMSE:', rmse_lm)
print('R2:', r2_lm)

RMSE: 0.07170052586570551
R2: 0.009869149178122538


In [29]:
monthly_ic = test.groupby('Date').apply(
    lambda x: x['Predicted Returns'].corr(x['Future Returns']),
    include_groups=False
)

print("LM Mean Monthly IC:", monthly_ic.mean())
print("LM Monthly IC St. Dev.:", monthly_ic.std())

LM Mean Monthly IC: 0.14287635359271136
LM Monthly IC St. Dev.: 0.3858118443147968


In [30]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

val_years = [2020, 2021, 2022, 2023]
alphas = [0.01, 0.1, 1.0, 10, 100, 1000, 10000, 100000]

results = []

for alpha in alphas:
    rmses = []
    for year in val_years:
        train_fold = data[data['Date'].dt.year < year]
        val_fold = data[data['Date'].dt.year == year]

        ridge = make_pipeline(
            StandardScaler(),
            Ridge(alpha=alpha)
        )

        ridge.fit(train_fold[features], train_fold['Future Returns'])
        preds_ridge = ridge.predict(val_fold[features])

        rmse = np.sqrt(mean_squared_error(val_fold['Future Returns'], preds_ridge))
        rmses.append(rmse)

    results.append({
        "alpha": alpha,
        "mean_val_rmse": np.mean(rmses)
    })

results_df = pd.DataFrame(results)
print(results_df.sort_values("mean_val_rmse"))
    

       alpha  mean_val_rmse
5    1000.00       0.094705
6   10000.00       0.094863
7  100000.00       0.094922
4     100.00       0.094925
3      10.00       0.095050
2       1.00       0.095066
1       0.10       0.095068
0       0.01       0.095068


In [32]:
ridge = make_pipeline(
        StandardScaler(),
        Ridge(alpha=1000)
    )

ridge.fit(X_train, y_train)
preds_ridge = ridge.predict(X_test)
test['Predicted Returns_Ridge'] = preds_ridge

In [33]:
rmse_ridge = np.sqrt(mean_squared_error(y_test, preds_ridge))
r2_ridge = r2_score(y_test, preds_ridge)

print('RMSE:', rmse_ridge)
print('R2:', r2_ridge)

RMSE: 0.07157975830033454
R2: 0.013201760429288


There is a slightly lower RMSE (-0.0002) and slightly higher R2 (+0.0034) in the Ridge regression model than the baseline linear model.

In [34]:
mic_ridge = test.groupby('Date').apply(
    lambda x: x['Predicted Returns - Ridge'].corr(x['Future Returns']),
    include_groups=False
)

print("Ridge Mean Monthly IC:", mic_ridge.mean())
print("Ridge Monthly IC St. Dev.:", mic_ridge.std())

Ridge Mean Monthly IC: 0.14728101376308622
Ridge Monthly IC St. Dev.: 0.3918356287829286
